# Get Five Below locations

#### Load Python tools and Jupyter config

In [1]:
import re
import us
import json
import asyncio
import aiohttp
import requests
import pandas as pd
import jupyter_black
import altair as alt
import geopandas as gpd
from pathlib import Path
from vega_datasets import data
from tqdm.asyncio import tqdm

In [2]:
jupyter_black.load()
pd.options.display.max_columns = 100
pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = None
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
place = "five-below"
place_formal = "Five Below"
color = "#0068b3"
today = pd.Timestamp.today().strftime("%Y-%m-%d")

---

## Scrape

#### Get every store page URL from the sitemap

The locator site is organized as directory > state > city > store, but its sitemap lists every page, so one request replaces the whole crawl. Store pages are the ones with three path segments: `/{state}/{city}/{street}`.

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
}

sitemap_xml = requests.get(
    "https://locations.fivebelow.com/sitemap1.xml", headers=headers
).text

store_url_pattern = re.compile(
    r"^https://locations\.fivebelow\.com/[a-z]{2}/[^/]+/[^/]+$"
)
store_urls = [
    url
    for url in re.findall(r"<loc>(.*?)</loc>", sitemap_xml)
    if store_url_pattern.match(url)
]

len(store_urls)

1977

#### Parse a store page

Each store page embeds two JSON-LD blocks: a schema.org `Store` node with the address, phone and hours, and a Yext "certified fact" block with exact coordinates and the full street address (including suite lines).

In [5]:
jsonld_pattern = re.compile(
    r'<script type="application/ld\+json">(.*?)</script>', re.S
)

DAY_ABBR = {
    "Monday": "Mon",
    "Tuesday": "Tue",
    "Wednesday": "Wed",
    "Thursday": "Thu",
    "Friday": "Fri",
    "Saturday": "Sat",
    "Sunday": "Sun",
}


def format_hours(specs):
    """Flatten openingHoursSpecification into 'Mon-Sat 09:30-21:30; Sun 10:00-20:00'."""
    if not specs:
        return None
    parts = []
    for spec in specs:
        days = spec.get("dayOfWeek", [])
        if isinstance(days, str):
            days = [days]
        days = [DAY_ABBR.get(d.rsplit("/", 1)[-1], d) for d in days]
        day_label = f"{days[0]}-{days[-1]}" if len(days) > 1 else days[0]
        parts.append(f"{day_label} {spec.get('opens')}-{spec.get('closes')}")
    return "; ".join(parts)


def parse_store(html, url):
    store, cert = None, None
    for block in jsonld_pattern.findall(html):
        try:
            parsed = json.loads(block)
        except json.JSONDecodeError:
            continue
        if "@graph" in parsed:
            for node in parsed["@graph"]:
                if node.get("@type") == "Store":
                    store = node
        elif "credentialSubject" in parsed:
            cert = parsed["credentialSubject"]

    if store is None:
        return None

    address = store.get("address", {})
    geo = (cert or {}).get("geo", {})

    # The cert block's streetAddress includes secondary lines (e.g. suite numbers)
    street_lines = (cert or {}).get("address", {}).get("streetAddress", "").split("\n")

    store_id = store.get("@id", "")
    id_match = re.search(r"#(\d+)-Store", store_id)

    return {
        "store_id": id_match.group(1) if id_match else None,
        "street": address.get("streetAddress"),
        "street2": street_lines[1] if len(street_lines) > 1 else None,
        "city": address.get("addressLocality"),
        "state": address.get("addressRegion"),
        "zip": address.get("postalCode"),
        "phone": store.get("telephone"),
        "latitude": geo.get("latitude"),
        "longitude": geo.get("longitude"),
        "hours": format_hours(store.get("openingHoursSpecification")),
        "url": url,
    }

#### Fetch all store pages concurrently

A semaphore caps concurrency at 16 requests to keep the load polite, with a couple of retries for transient failures.

In [6]:
semaphore = asyncio.Semaphore(16)


async def fetch_store(session, url):
    async with semaphore:
        for attempt in range(3):
            try:
                async with session.get(url) as response:
                    if response.status == 200:
                        return parse_store(await response.text(), url)
                    if response.status == 404:
                        return None
            except (aiohttp.ClientError, asyncio.TimeoutError):
                pass
            await asyncio.sleep(2**attempt)
    return None


async with aiohttp.ClientSession(
    headers=headers, timeout=aiohttp.ClientTimeout(total=30)
) as session:
    results = await tqdm.gather(*[fetch_store(session, url) for url in store_urls])

rows = [row for row in results if row]
print(f"Parsed {len(rows)} of {len(store_urls)} store pages")

  0%|          | 0/1977 [00:00<?, ?it/s]

  0%|          | 2/1977 [00:00<02:00, 16.35it/s]

  1%|          | 12/1977 [00:00<00:34, 57.69it/s]

  1%|▏         | 25/1977 [00:00<00:22, 86.76it/s]

  2%|▏         | 37/1977 [00:00<00:20, 96.10it/s]

...

 98%|█████████▊| 1935/1977 [00:16<00:00, 121.48it/s]

 99%|█████████▊| 1949/1977 [00:16<00:00, 122.66it/s]

 99%|█████████▉| 1967/1977 [00:16<00:00, 134.30it/s]

100%|██████████| 1977/1977 [00:17<00:00, 115.35it/s]

Parsed 1975 of 1977 store pages


In [7]:
df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset="store_id")
    .sort_values(["state", "city", "street"])
    .reset_index(drop=True)
)
len(df)

1975

#### Full state names, brand and fetch date

In [8]:
state_mapping = {state.abbr: state.name for state in us.states.STATES_AND_TERRITORIES}
df["state_name"] = df["state"].map(state_mapping)
df["brand"] = place_formal
df["updated"] = today

In [9]:
df.head()

,store_id,street,street2,city,state,zip,phone,latitude,longitude,hours,url,state_name,brand,updated
0,22747233,100 S Colonial Dr.,Suite 2350,Alabaster,AL,35007,+12055641095,33.226698,-86.804772,Mon-Sat 09:30-21:30; Sun 10:00-20:00,https://locations.fivebelow.com/al/alabaster/100-s-colonial-dr.,Alabama,Five Below,2026-07-16
1,20321065,7200 US Highway 431,Suite 100,Albertville,AL,35950,+12566661330,34.276891,-86.204849,Mon-Sat 09:30-21:30; Sun 10:00-20:00,https://locations.fivebelow.com/al/albertville/7200-us-highway-431,Alabama,Five Below,2026-07-16
2,1076890487,4782 Hwy. 280,NaN,Alexander City,AL,35010,+12563051109,32.916403,-85.953609,Mon-Sat 09:30-21:30; Sun 10:00-20:00,https://locations.fivebelow.com/al/alexander-city/4782-hwy.-280,Alabama,Five Below,2026-07-16
3,63653432,140 Covington Mall Drive,NaN,Andalusia,AL,36420,+13342615178,31.316754,-86.496321,Mon-Sat 09:30-21:30; Sun 10:00-20:00,https://locations.fivebelow.com/al/andalusia/140-covington-mall-drive,Alabama,Five Below,2026-07-16
4,1047114440,1061 Kelli Drive,Unit E,Athens,AL,35611,+12562853105,34.784114,-86.935031,Mon-Sat 09:30-21:30; Sun 10:00-20:00,https://locations.fivebelow.com/al/athens/1061-kelli-drive,Alabama,Five Below,2026-07-16


---

## Geography

#### Make it a geodataframe

In [10]:
gdf = gpd.GeoDataFrame(
    df.copy(), geometry=gpd.points_from_xy(df.longitude, df.latitude)
).set_crs("4326")

---

## Maps

#### US states background

In [11]:
background = (
    alt.Chart(alt.topo_feature(data.us_10m.url, feature="states"))
    .mark_geoshape(fill="#e9e9e9", stroke="white")
    .properties(width=800, height=500, title=f"{place_formal} locations")
    .project("albersUsa")
)

#### Location points map

In [12]:
points = (
    alt.Chart(gdf)
    .mark_circle(size=5, color=color)
    .encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
    )
)

point_map = background + points
point_map.configure_view(stroke=None)

alt.LayerChart(...)

#### Location proportional symbols map

In [13]:
symbols = (
    alt.Chart(gdf)
    .transform_aggregate(
        latitude="mean(latitude)",
        longitude="mean(longitude)",
        count="count()",
        groupby=["state"],
    )
    .mark_circle()
    .encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
        size=alt.Size("count:Q", title="Count by state"),
        color=alt.value(color),
        tooltip=["state:N", "count:Q"],
    )
    .properties(
        title=f"Number of {place_formal} in US, by average lon/lat of locations"
    )
)

symbol_map = background + symbols
symbol_map.configure_view(stroke=None)

alt.LayerChart(...)

---

## Exports

#### JSON, CSV and GeoJSON

In [14]:
Path("data/processed").mkdir(parents=True, exist_ok=True)

df.to_json(
    f"data/processed/{place}_locations.json",
    indent=4,
    orient="records",
)
df.to_csv(f"data/processed/{place}_locations.csv", index=False)
gdf.to_file(f"data/processed/{place}_locations.geojson", driver="GeoJSON")
gdf.to_file(f"data/processed/{place}_locations_{today}.geojson", driver="GeoJSON")